## **Setup**

In [30]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from neo4j import GraphDatabase

# **Data Loading**

In [31]:
DATA_DIR = './dataset/'
FOLDER2VAR = {
    "Watt W": "watt", "Apperent power VA": "va", "Current A": "current", "Voltage V": "voltage",
    "KWh/Today": "kwh_today", "KWh/Total": "kwh_total",
    "Temperature": "temp", "Humidity": "hum", "Motion": "motion",
}

def parse_entity(fname):
    """
    Retrieve device name from the dataset csv file
    Example:
    "2024-06-21-16-57 Chronograf Data Coffee Machine.csv" -> "Coffee Machine"

    Returns string
    """
    n = os.path.splitext(fname)[0]
    n = re.sub(r"^\d{4}-\d{2}-\d{2}-\d{2}-\d{2}\s*", "", n)
    return n.replace("Chronograf Data", "").strip(" -")

frames = []
for csv in sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True)):
    rel = os.path.relpath(csv, DATA_DIR).replace("\\", "/")
    folder = "/".join(rel.split("/")[:-1])

    df = pd.read_csv(csv)
    df.columns = ["time", "value"]
    df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["variable"] = FOLDER2VAR[folder]
    df["entity"] = parse_entity(os.path.basename(csv))
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
raw.head()

,time,value,variable,entity
0,2024-01-01 14:12:00,NaN,va,Coffee Machine
1,2024-01-02 01:36:00,NaN,va,Coffee Machine
2,2024-01-02 13:00:00,NaN,va,Coffee Machine
3,2024-01-03 00:24:00,NaN,va,Coffee Machine
4,2024-01-03 11:48:00,6.518519,va,Coffee Machine


In [32]:
main =  raw[raw["entity"] != "GENERAL"]

def wide(var):
    return main[main["variable"] == var].pivot(index="time", columns="entity", values="value").sort_index()

WATT, VA, TEMP, HUM, MOTION = (wide(v) for v in ["watt", "va", "temp", "hum", "motion"])
ALL_DEVICES = ["Coffee Machine", "Desktop", "Fridge", "Kettle", "Microwave", "Printer", "Water Dispenser"]
print("Bentuk tabel VA:", VA.shape, "| kolom:", list(VA.columns))

Bentuk tabel VA: (361, 7) | kolom: ['Coffee Machine', 'Desktop', 'Fridge', 'Kettle', 'Microwave', 'Printer', 'Water Dispenser']


In [33]:
main.head()

,time,value,variable,entity
0,2024-01-01 14:12:00,NaN,va,Coffee Machine
1,2024-01-02 01:36:00,NaN,va,Coffee Machine
2,2024-01-02 13:00:00,NaN,va,Coffee Machine
3,2024-01-03 00:24:00,NaN,va,Coffee Machine
4,2024-01-03 11:48:00,6.518519,va,Coffee Machine


In [34]:
ELECTRICAL = {"watt", "va", "current", "voltage", "kwh_today", "kwh_total"}
UNIT = {"watt": "W", "va": "VA", "current": "A", "voltage": "V", "kwh_today": "kWh", "kwh_total": "kWh"}

def area_of(entity):
    return "Lab" if entity.startswith("Lab") else entity

def sensor_type(var, entity):
    return "door" if (var == "motion" and "door" in entity.lower()) else var

def build_payload(raw):
    
    areas, devices, sensors = set(), {}, []
    for (var, entity), g in raw.groupby(["variable", "entity"]):
        sid = f"{var}:{entity}"
        is_dev = var in ELECTRICAL
        owner = entity if is_dev else area_of(entity)
        if is_dev:
            devices[entity] = "general_meter" if entity == "GENERAL" else "appliance"
        else:
            areas.add(owner)
        sensors.append({"id": sid, 
                        "type": sensor_type(var, entity), 
                        "unit": UNIT.get(var),
                        "label": entity, 
                        "owner": owner, 
                        "owner_kind": 
                        "device" if is_dev else "area",
                        })

    valid = raw.dropna(subset=["value"])
    readings = [{"sensor_id": f"{v}:{e}", "time": t.isoformat(), "value": float(x)}
                for v, e, t, x in zip(valid["variable"], valid["entity"], valid["time"], valid["value"])]
    return {"areas": sorted(areas), "devices": devices, "sensors": sensors, "readings": readings}

building_payload = build_payload(raw)


# **Building Neo4j Graph**

In [ ]:
def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]


In [ ]:
# local
driver = GraphDatabase.driver(
    "neo4j://localhost:7687",
    auth=("neo4j", "neo4j123")
)

In [ ]:
SCHEMA = [
    "CREATE CONSTRAINT area_name IF NOT EXISTS FOR (a:Area) REQUIRE a.name IS UNIQUE",
    "CREATE CONSTRAINT device_name IF NOT EXISTS FOR (d:Device) REQUIRE d.name IS UNIQUE",
    "CREATE CONSTRAINT sensor_id IF NOT EXISTS FOR (s:Sensor) REQUIRE s.id IS UNIQUE",
    "CREATE INDEX reading_time IF NOT EXISTS FOR (r:Reading) ON (r.time)",
    "CREATE CONSTRAINT reading_identity IF NOT EXISTS FOR (r:Reading) REQUIRE (r.sensor_id, r.time) IS UNIQUE",
    "CREATE CONSTRAINT timepoint_time IF NOT EXISTS FOR (t:TimePoint) REQUIRE t.time IS UNIQUE",
]


with driver.session(database="ts-smart-building") as session:

    for stmt in SCHEMA:
        session.run(stmt)

    dev_rows = [{"name": k, "kind": v} for k, v in building_payload["devices"].items()]
    dev_sensors = [s for s in building_payload["sensors"] if s["owner_kind"] == "device"]
    area_sensors = [s for s in building_payload["sensors"] if s["owner_kind"] == "area"]
    link_next = True
    batch_size = 64

    session.run("""
    UNWIND $devices AS dev
    MERGE (d:Device {name: dev.name})
    SET d.kind = dev.kind
    """, devices=dev_rows)

    session.run("""
    UNWIND $sensors AS s
    MERGE (se:Sensor {id: s.id})
    SET
        se.type = s.type,
        se.unit = s.unit,
        se.label = s.label
    """, sensors=building_payload["sensors"])

    session.run("""
    UNWIND $sensors AS s
    MATCH (d:Device {name: s.owner}),
          (se:Sensor {id: s.id})
    MERGE (d)-[:COLLECTS]->(se)
    """, sensors=dev_sensors)

    session.run("""
    UNWIND $sensors AS s
    MATCH (a:Area {name: s.owner}),
          (se:Sensor {id: s.id})
    MERGE (a)-[:HAS_SENSOR]->(se)
    """, sensors=area_sensors)

    total = len(building_payload["readings"])
    for i, batch in enumerate(chunks(building_payload["readings"], batch_size), 1):

        session.run("""
        UNWIND $rows AS row
        MATCH (se:Sensor {id: row.sensor_id})
        MERGE (r:Reading {
            sensor_id: row.sensor_id,
            time: datetime(row.time)
        })
        SET r.value = row.value
        MERGE (se)-[:READS]->(r)
        """, rows=batch)

        print(f"  readings: {min(i * batch_size, total):,}/{total:,}")

    print()

    for i, batch in enumerate(chunks(building_payload["readings"], batch_size), 1):

        session.run("""
        UNWIND $rows AS row
        MATCH (r:Reading {
            sensor_id: row.sensor_id,
            time: datetime(row.time)
        })
        MERGE (tp:TimePoint {
            time: datetime(row.time)
        })
        MERGE (r)-[:AT_TIME]->(tp)
        """, rows=batch)

        print(f"timepoints: {min(i * batch_size, total):,}/{total:,}")
    print()

    if link_next:
        for s in building_payload["sensors"]:
            session.run("""
            MATCH (se:Sensor {id: $sensor_id})-[:READS]->(r:Reading)
            WITH r
            ORDER BY r.time
            WITH collect(r) AS rs
            UNWIND range(0, size(rs) - 2) AS i
            WITH rs[i] AS a, rs[i + 1] AS b
            MERGE (a)-[:NEXT]->(b)
            """, sensor_id=s["id"])


  readings: 64/17,003
  readings: 128/17,003
  readings: 192/17,003
  readings: 256/17,003
  readings: 320/17,003
  readings: 384/17,003
  readings: 448/17,003
  readings: 512/17,003
  readings: 576/17,003
  readings: 640/17,003
  readings: 704/17,003
  readings: 768/17,003
  readings: 832/17,003
  readings: 896/17,003
  readings: 960/17,003
  readings: 1,024/17,003
  readings: 1,088/17,003
  readings: 1,152/17,003
  readings: 1,216/17,003
  readings: 1,280/17,003
  readings: 1,344/17,003
  readings: 1,408/17,003
  readings: 1,472/17,003
  readings: 1,536/17,003
  readings: 1,600/17,003
  readings: 1,664/17,003
  readings: 1,728/17,003
  readings: 1,792/17,003
  readings: 1,856/17,003
  readings: 1,920/17,003
  readings: 1,984/17,003
  readings: 2,048/17,003
  readings: 2,112/17,003
  readings: 2,176/17,003
  readings: 2,240/17,003
  readings: 2,304/17,003
  readings: 2,368/17,003
  readings: 2,432/17,003
  readings: 2,496/17,003
  readings: 2,560/17,003
  readings: 2,624/17,003
  read

In [ ]:
driver.close()

# **Query di Aura**

Note: ini datanya dari hasil export dump dari graph database local terus restore di cloud.

```sql
-- bikin graph projection (pastikan ini harus dijalankan sebelum melakukan clustering)
MATCH (s:Sensor)-[:READS]->(r:Reading)
WHERE s.type = 'va'
  AND r.hdbscan_features IS NOT NULL

WITH gds.graph.project(
    'energyReadings',
    r,
    null,
    {
        sourceNodeLabels: ['Reading'],
        sourceNodeProperties: r {
            .hdbscan_features
        }
    },
    {memory: '2GB'}
) AS g

RETURN
    g.graphName AS graphName,
    g.nodeCount AS nodeCount,
    g.relationshipCount AS relationshipCount;
```

```sql
MATCH (d:Device {name: 'Fridge'})
      -[:COLLECTS]->(s:Sensor)
      -[:READS]->(r:Reading)

WHERE s.unit = 'VA'
  AND r.value IS NOT NULL

WITH s, avg(r.value) AS mean, stDev(r.value) AS std

MATCH (s)-[:READS]->(r:Reading)
WHERE r.value IS NOT NULL

SET r.hdbscan_features = [
    CASE
        WHEN std IS NULL OR std = 0 THEN 0.0
        ELSE (r.value - mean) / std
    END
]

RETURN count(r) AS updated_readings;

CALL gds.kmeans.stream('energyReading', {
    nodeProperty: 'hdbscan_features',
    k: 2,
    randomSeed: 42
})
YIELD nodeId, communityId

WITH gds.util.asNode(nodeId) AS r, communityId

MATCH (d:Device {name: 'Fridge'})
      -[:COLLECTS]->(s:Sensor)
      -[:READS]->(r:Reading)

RETURN
    s.id AS sensor_id,
    s.type AS sensor_type,
    r.time AS reading_time,
    r.value AS value,
    communityId AS cluster_id,
    CASE
        WHEN communityId = -1 THEN true
        ELSE false
    END AS potential_anomaly

ORDER BY cluster_id, reading_time;

CALL gds.hdbscan.stream('energyReading', {
    nodeProperty: 'hdbscan_features',
    minClusterSize: 2,
    samples: 2
})
YIELD nodeId, label

WITH gds.util.asNode(nodeId) AS r, label

MATCH (d:Device {name: 'Fridge'})
      -[:COLLECTS]->(s:Sensor)
      -[:READS]->(r:Reading)

SET r.hdbscan_cluster = label

RETURN
    s.id AS sensor_id,
    s.type AS sensor_type,
    r.time AS reading_time,
    r.value AS value,
    label AS cluster_id,
    CASE
        WHEN label = -1 THEN true
        ELSE false
    END AS potential_anomaly


ORDER BY cluster_id, reading_time;

MATCH (d:Device)-[:COLLECTS]->(s:Sensor)
      -[:READS]->(r:Reading)

WHERE r.hdbscan_cluster = -1

OPTIONAL MATCH (prev:Reading)-[:NEXT]->(r)
OPTIONAL MATCH (r)-[:NEXT]->(next:Reading)

RETURN
    d.name AS device,
    prev.time AS previous_time,
    prev.value AS previous_power,
    r.time AS anomaly_time,
    r.value AS anomaly_power,
    next.time AS next_time,
    next.value AS next_power

ORDER BY device, anomaly_time;
```

In [ ]:
import pandas as pd

def create_pivot(path_csv: str, index: str):
    
    df = pd.read_csv(path_csv)
    pivot = pd.pivot_table(
        df,
        index=index,
        values="value",
        aggfunc=["count", "min", "max", "mean"],
    )

    pivot.columns = ["count", "min_value", "max_value", "mean_value"]
    return pivot

devices = ['coffee', 'desk', 'dispenser', 'fridge', 'microwave', 'kettle', 'print']

algorithms = ['kmeans', 'hdbscan']
indexes = ['cluster_id', 'potential_anomaly']

for algo, index in zip(algorithms, indexes):
    print(algo)
    fin = []
    for device in devices:
        res = create_pivot(f'./result/{device}_{algo}.csv', index)
        print(res)
        fin.append([device]+res['count'].tolist() + res['min_value'].tolist() +res['max_value'].tolist())
    sv = pd.DataFrame(fin, columns=['name', 'count (A)', 'count (B)', 'min (A)', 'min (B)', 'max (A)', 'max (B)'])
    sv.to_csv(f"result/{algo}_analysis.csv")

kmeans
            count  min_value   max_value  mean_value
cluster_id                                          
0             263   3.571429   25.577982    8.148521
1              80  25.881356  108.631579   57.079686
            count   min_value    max_value  mean_value
cluster_id                                            
0              84    0.000000   624.333333  176.177558
1              76  630.333333  1282.333333  887.906224
            count   min_value    max_value  mean_value
cluster_id                                            
0             248    1.636364   293.592593   25.206936
1              98  304.000000  1681.250000  668.171000
            count  min_value   max_value  mean_value
cluster_id                                          
0             238   0.000000   69.120879   60.602419
1             122  69.168675  101.246753   77.463536
            count  min_value  max_value  mean_value
cluster_id                                         
0             243   8.184